# 06 — Crosswalk and master table

**Owner:** F  
**Goal:** one row per municipality per election, with the funnel columns

**Reads:** `data/interim/*.csv`  
**Writes:** `PROCESSED / "master.csv"`

Run the two setup cells below first, every time.

## Setup

In [27]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

Repo: C:\Users\27664\Desktop\Projects\democratic-funnel


In [28]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

Environment: Local (repo)
Repo root:   C:\Users\27664\Desktop\Projects\democratic-funnel
Work root:   C:\Users\27664\Desktop\Projects\democratic-funnel


## Crosswalk

Codes present in one year but not another — map them by hand, with a note per row.

In [ ]:
# cw = pd.read_csv(INTERIM / "muni_crosswalk.csv")
# set(df11.muni_code) - set(df16.muni_code)

In [29]:
cen = pd.read_csv(INTERIM / "census_2022.csv")
reg = pd.read_csv(INTERIM / "registration_2026.csv")

m = cen.merge(reg, on="muni_code")
m["reg_youth"] = m["Total_18 - 19"] + m["Total_20 - 29"]

EA, EY = m.eligible_adults.sum(), m.eligible_youth.sum()
R,  RY = m.Total_Registered_Voters.sum(), m.reg_youth.sum()

print(f"eligible adults        {EA:>14,.0f}")
print(f"eligible youth 18-29   {EY:>14,.0f}   (expect ~12-13m)")
print(f"youth share of adults  {EY/EA:>14.3f}   (expect ~0.31-0.33)")
print()
print(f"registration rate      {R/EA:>14.3f}")
print(f"  youth                {RY/EY:>14.3f}")
print(f"  everyone else        {(R-RY)/(EA-EY):>14.3f}   (MUST be < 1.0)")
print()
ry = m.reg_youth / m.eligible_youth
print("per-municipality youth rate:")
print(ry.describe().round(3).to_string())
print(f"\nmunicipalities with youth rate > 1: {(ry > 1).sum()}")

eligible adults            40,050,279
eligible youth 18-29       12,005,388   (expect ~12-13m)
youth share of adults           0.300   (expect ~0.31-0.33)

registration rate               0.730
  youth                         0.462
  everyone else                 0.845   (MUST be < 1.0)

per-municipality youth rate:
count    213.000
mean       0.557
std        0.162
min        0.265
25%        0.452
50%        0.545
75%        0.621
max        1.835

municipalities with youth rate > 1: 3


## Merge everything on muni_code

Results + registration + signals + census. Left join on results.

In [ ]:
# master = (results.merge(reg, on="muni_code", how="left")
#                  .merge(signals, on="muni_code", how="left")
#                  .merge(census, on="muni_code", how="left"))

In [30]:
import numpy as np, pandas as pd, geopandas as gpd

results = pd.read_csv(INTERIM / "results_lge_2011_2021.csv")
reg     = pd.read_csv(INTERIM / "registration_2026.csv")
cen     = pd.read_csv(INTERIM / "census_2022.csv")
signals = pd.read_csv(INTERIM / "recent_signals.csv")     # 2024 national election + by-elections (03)
nb      = pd.read_csv(INTERIM / "neighbours.csv")
gdf     = gpd.read_file(INTERIM / "boundaries.gpkg")

# 2026 row: registration is known, turnout is what we predict
row26 = reg.rename(columns={"Total_Registered_Voters": "registered"}).copy()
row26["registered_youth"] = row26["Total_18 - 19"] + row26["Total_20 - 29"]
row26["election"] = 2026
row26 = row26[["election", "province", "muni_code", "muni_name",
               "registered", "registered_youth"]]

master = pd.concat([results, row26], ignore_index=True)
master = master.merge(cen, on="muni_code", how="left")
# recent signals are one value per municipality (like the Census), repeated on every election row
master = master.merge(signals[["muni_code", "npe2024_turnout", "byelection_turnout_mean", "byelection_count"]],
                      on="muni_code", how="left")
master = master.merge(gdf.drop(columns="geometry")[
            ["muni_code", "muni_category", "district_code", "area_km2"]],
            on="muni_code", how="left")

print(master.shape)                                  # expect (852, ~23)
print(master.groupby("election").size())

(852, 23)
election
2011    213
2016    213
2021    213
2026    213
dtype: int64


In [31]:
master["r"] = master["registered"] / master["eligible_adults"]
master["t"] = master["votes_cast"] / master["registered"]
master["p"] = master["r"] * master["t"]

# leaks against that year's national median
for yr, g in master.groupby("election"):
    master.loc[g.index, "leak_reg"]  = np.log(g["r"] / g["r"].median())
    master.loc[g.index, "leak_turn"] = np.log(g["t"] / g["t"].median())

master["pop_density"] = master["eligible_adults"] / master["area_km2"]

# model target: the CHANGE in turnout, not the level
master = master.sort_values(["muni_code", "election"])
master["t_prev"]  = master.groupby("muni_code")["t"].shift(1)
master["r_prev"]  = master.groupby("muni_code")["r"].shift(1)
master["dt"]      = master["t"] - master["t_prev"]

# neighbours' turnout, same year
lag = (nb.merge(master[["muni_code", "election", "t"]],
                left_on="muni_code_right", right_on="muni_code")
         .groupby(["muni_code_left", "election"])["t"].mean()
         .rename("spatial_lag_turnout").reset_index()
         .rename(columns={"muni_code_left": "muni_code"}))
master = master.merge(lag, on=["muni_code", "election"], how="left")

print(master.groupby("election")[["r", "t", "p"]].median().round(3))

              r      t      p
election                     
2011      0.630  0.595  0.371
2016      0.696  0.587  0.402
2021      0.696  0.482  0.336
2026      0.787    NaN    NaN


In [32]:
print("r > 1.05 (Census undercount - flag, don't drop):")
print(master.loc[master.r > 1.05, ["election","muni_code","muni_name","r"]].to_string(index=False))

print("\nmissing values:")
print(master.isna().sum()[lambda s: s > 0].to_string())

master.to_csv(PROCESSED / "master.csv", index=False)
print("\nsaved", master.shape)

r > 1.05 (Census undercount - flag, don't drop):
 election muni_code    muni_name        r
     2026    KZN226 Mkhambathini 1.100916
     2026    KZN286      Nkandla 1.088022
     2026    LIM361   Thabazimbi 1.157470
     2026     NC062    NAMA KHOI 1.107249
     2011     NC064   Kamiesberg 1.104999
     2016     NC064   KAMIESBERG 1.168420
     2021     NC064   KAMIESBERG 1.192154
     2026     NC064   KAMIESBERG 1.228633
     2011     NC067      Khâi-Ma 1.654952
     2016     NC067      KHâI-MA 1.817354
     2021     NC067      KHâI-MA 1.988154
     2026     NC067      KHâI-MA 2.336635
     2026     NC071       UBUNTU 1.060473
     2016     NC074    KAREEBERG 1.073254
     2021     NC074    KAREEBERG 1.088273
     2026     NC074    KAREEBERG 1.173619
     2026     NC084       !KHEIS 1.052501
     2026     NC453     GAMAGARA 1.230015

missing values:
valid_votes            213
spoilt                 213
votes_cast             213
turnout                213
registered_youth       639
t

In [33]:
master["r_implausible"] = master["r"] > 1.0
print(master.groupby("election")["r_implausible"].sum())

# check the worst case directly
print(master[master.muni_code == "NC067"][
      ["election","muni_name","registered","eligible_adults","r"]].to_string(index=False))

election
2011     3
2016     4
2021     4
2026    15
Name: r_implausible, dtype: int64
 election muni_name  registered  eligible_adults        r
     2011   Khâi-Ma 6491.937992      3922.735082 1.654952
     2016   KHâI-MA 7129.000000      3922.735082 1.817354
     2021   KHâI-MA 7799.000000      3922.735082 1.988154
     2026   KHâI-MA 9166.000000      3922.735082 2.336635


## Drop districts, keep district_code

Districts would double-count voters.

In [ ]:
# master = master[master["muni_type"] != "DC"]

## The funnel

r = R/E, t = V/R, p = r*t, then the two leaks against the national benchmark.

In [ ]:
import numpy as np
# master["r"] = master["registered"] / master["eligible_adults"]
# master["t"] = master["votes_cast"] / master["registered"]
# master["p"] = master["r"] * master["t"]
# r_star, t_star = master["r"].median(), master["t"].median()
# master["leak_reg"]  = np.log(master["r"] / r_star)
# master["leak_turn"] = np.log(master["t"] / t_star)

In [34]:
master["r_implausible"] = master["r"] > 1.0
print(master.groupby("election")["r_implausible"].sum())

# check the worst case directly
print(master[master.muni_code == "NC067"][
      ["election","muni_name","registered","eligible_adults","r"]].to_string(index=False))

election
2011     3
2016     4
2021     4
2026    15
Name: r_implausible, dtype: int64
 election muni_name  registered  eligible_adults        r
     2011   Khâi-Ma 6491.937992      3922.735082 1.654952
     2016   KHâI-MA 7129.000000      3922.735082 1.817354
     2021   KHâI-MA 7799.000000      3922.735082 1.988154
     2026   KHâI-MA 9166.000000      3922.735082 2.336635
